In [2]:
import numpy as np
import pandas as pd
import os

from lifelines import KaplanMeierFitter
from lifelines.utils import restricted_mean_survival_time
from tqdm import tqdm
import warnings
from scipy.integrate import IntegrationWarning
warnings.filterwarnings("ignore", category=IntegrationWarning)

In [3]:
def loo_rmst(df_group, rmst_group, follow_up, time, event):
    rmst_list = []
    group_size = len(df_group)
    for i in range(group_size):
        idx_to_drop = df_group.index[i]
        df_loo = df_group.drop(index=idx_to_drop)
        kmf = KaplanMeierFitter().fit(df_loo[time], event_observed=df_loo[event])
        rmst = group_size * rmst_group - (group_size-1) * restricted_mean_survival_time(kmf, t=follow_up, return_variance=False)
        rmst_list.append(rmst)
    return rmst_list

## Simulate df with varying treatment effect

In [ ]:
n_values = [30, 100, 500]
fc = 0.4
follow_up = 1.5
censore_rate_pred = 0.10

for ve in [10,20,30,40,60,80]:
        df_sim = []
        estimations = []
        for n in n_values:
                folder_path = f"./sim_ct_df_with_censore_weibull/sim_n{n}_fc{fc}_ve{ve}_censpred{censore_rate_pred}"
                files = sorted(os.listdir(folder_path))
                for file in tqdm(files, desc=f"Simulations for n={n} and vaccine efficacy={ve}%"):
                        df = pd.read_csv(os.path.join(folder_path, file))
                        df["n"] = n
                        df["sim"] = file.split('_')[1].split('.')[0]

                        # Compute outcome: leave-one-out RMST for each patient
                        kmf_trt = KaplanMeierFitter().fit(df[df['group']==1]['time_obs'], event_observed=df[df['group']==1]['event_obs'])
                        kmf_con = KaplanMeierFitter().fit(df[df['group']==0]['time_obs'], event_observed=df[df['group']==0]['event_obs'])
                        rmst_placebo  = restricted_mean_survival_time(kmf_con, t=1.5, return_variance=False)
                        rmst_active = restricted_mean_survival_time(kmf_trt, t=1.5, return_variance=False)

                        df.loc[df['group'] == 1, 'LOO_RMST_SEP'] = loo_rmst(df[df['group'] == 1], rmst_active, follow_up, time='time_obs', event='event_obs')
                        df.loc[df['group'] == 0, 'LOO_RMST_SEP'] = loo_rmst(df[df['group'] == 0], rmst_placebo, follow_up, time='time_obs', event='event_obs')
                        var_placebo = df.loc[df['group']==0]['LOO_RMST_SEP'].var()
                        var_active = df.loc[df['group']==1]['LOO_RMST_SEP'].var()

                        # Compute classic estimator
                        ate_rmst_manual = df[df['group'] == 1]['LOO_RMST_SEP'].mean() - df[df['group'] == 0]['LOO_RMST_SEP'].mean()
                        var_ate_rmst_manual = var_active/n + var_placebo/n
                
                        i = 0.6
                        df_tmp = df[df['noise'] == round(i,2)].copy()
                        kmf_trt_pred = KaplanMeierFitter().fit(df_tmp[df_tmp['group']==1]['time_pred'], event_observed=df_tmp[df_tmp['group']==1]['event_pred'])
                        kmf_con_pred = KaplanMeierFitter().fit(df_tmp[df_tmp['group']==0]['time_pred'], event_observed=df_tmp[df_tmp['group']==0]['event_pred'])
                        rmst_active_pred = restricted_mean_survival_time(kmf_trt_pred, t=1.5, return_variance=False) 
                        rmst_placebo_pred = restricted_mean_survival_time(kmf_con_pred, t=1.5, return_variance=False)
                        df_tmp.loc[df_tmp['group'] == 1, 'PREDS_LOO_RMST_WEIBULL'] = loo_rmst(df_tmp[df_tmp['group'] == 1], rmst_active_pred, follow_up, time='time_pred', event='event_pred')
                        df_tmp.loc[df_tmp['group'] == 0, 'PREDS_LOO_RMST_WEIBULL'] = loo_rmst(df_tmp[df_tmp['group'] == 0], rmst_placebo_pred, follow_up, time='time_pred', event='event_pred')
                        df_tmp['noise_level'] = round(i,2)
                        
                        # Compute PPCT estimator
                        sigma_f_2 = df_tmp['PREDS_LOO_RMST_WEIBULL'].var()
                        sigma_t_2 = var_active
                        sigma_c_2 = var_placebo
                        rho_t = np.cov(df_tmp[df_tmp['group'] == 1]['PREDS_LOO_RMST_WEIBULL'], df_tmp[df_tmp['group'] == 1]['LOO_RMST_SEP'])[0, 1]/np.sqrt(sigma_f_2*sigma_t_2)
                        rho_c = np.cov(df_tmp[df_tmp['group'] == 0]['PREDS_LOO_RMST_WEIBULL'], df_tmp[df_tmp['group'] == 0]['LOO_RMST_SEP'])[0, 1]/np.sqrt(sigma_f_2*sigma_c_2)
                        lambda_star_shifted_order = (n * np.sqrt(sigma_t_2) * rho_t + n * np.sqrt(sigma_c_2) * rho_c) / ((n+n) * np.sqrt(sigma_f_2))
                        var_ppi_shifted_order = (1/n) * (sigma_t_2 + (lambda_star_shifted_order**2)*sigma_f_2 - 2*lambda_star_shifted_order*np.sqrt(sigma_f_2*sigma_t_2)*rho_t)  + \
                                ((1/n) * (sigma_c_2 + (lambda_star_shifted_order**2)*sigma_f_2 - 2*lambda_star_shifted_order*np.sqrt(sigma_f_2*sigma_c_2)*rho_c))
                        ate_ppi_shifted_order = (df_tmp[df_tmp['group'] == 1]['LOO_RMST_SEP']- lambda_star_shifted_order* df_tmp[df_tmp['group'] == 1]['PREDS_LOO_RMST_WEIBULL']).mean() - \
                                (df_tmp[df_tmp['group'] == 0]['LOO_RMST_SEP']- lambda_star_shifted_order* df_tmp[df_tmp['group'] == 0]['PREDS_LOO_RMST_WEIBULL']).mean()
                        r_2_shifted_order = df_tmp['LOO_RMST_SEP'].corr(df_tmp['PREDS_LOO_RMST_WEIBULL'])**2
                        var_r2_shifted_order = var_ate_rmst_manual * (1- r_2_shifted_order)

                        estimations.append({
                                "n": n,
                                "sim": file.split('_')[1].split('.')[0],
                                "noise_level": round(i,2),
                                "ate_classic RMST": ate_rmst_manual,
                                "var_classic RMST": var_ate_rmst_manual,
                                "ate_ppi_shifted_order RMST": ate_ppi_shifted_order,
                                "var_ppi_shifted_order RMST": var_ppi_shifted_order,
                                "r2_shifted_order RMST": r_2_shifted_order,
                                "lambda_star_shifted_order RMST": lambda_star_shifted_order,
                                "var_ppi_with_r2_formula_shifted_order RMST": var_r2_shifted_order,
                                })
                        df_sim.append(df_tmp)

        df_sim = pd.concat(df_sim, ignore_index=True)
        estimations = pd.DataFrame(estimations)
        df_sim.to_csv(f"sim_outputs_1000_censore_weibull/df_sim_fc{fc}_ve{ve}_censpred{censore_rate_pred}.csv", index=False)
        estimations.to_csv(f"sim_outputs_1000_censore_weibull/estimations_fc{fc}_ve{ve}_censpred{censore_rate_pred}.csv", index=False)


Simulations for n=100 and vaccine efficacy=30%:  23%|██▎       | 234/1000 [24:45<1:23:40,  6.55s/it]

## Simulate df with varying event probability in the control group

In [ ]:
n_values =[30, 100, 500]
ve = 30
follow_up = 1.5
censore_rate_pred = 0.1

for fc in [0.1, 0.2, 0.6, 0.8]: # 0.4 already done
        df_sim = []
        estimations = []
        for n in n_values:
                folder_path = f"./sim_ct_df_with_censore_weibull/sim_n{n}_fc{fc}_ve{ve}_censpred{censore_rate_pred}"
                files = sorted(os.listdir(folder_path))
                for file in tqdm(files, desc=f"Simulations for n={n} and vaccine efficacy={ve}%"):
                        df = pd.read_csv(os.path.join(folder_path, file))
                        df["n"] = n
                        df["sim"] = file.split('_')[1].split('.')[0]

                        # Compute outcome: leave-one-out RMST for each patient
                        kmf_trt = KaplanMeierFitter().fit(df[df['group']==1]['time_obs'], event_observed=df[df['group']==1]['event_obs'])
                        kmf_con = KaplanMeierFitter().fit(df[df['group']==0]['time_obs'], event_observed=df[df['group']==0]['event_obs'])
                        rmst_placebo  = restricted_mean_survival_time(kmf_con, t=1.5, return_variance=False)
                        rmst_active = restricted_mean_survival_time(kmf_trt, t=1.5, return_variance=False)

                        df.loc[df['group'] == 1, 'LOO_RMST_SEP'] = loo_rmst(df[df['group'] == 1], rmst_active, follow_up, time='time_obs', event='event_obs')
                        df.loc[df['group'] == 0, 'LOO_RMST_SEP'] = loo_rmst(df[df['group'] == 0], rmst_placebo, follow_up, time='time_obs', event='event_obs')
                        var_placebo = df.loc[df['group']==0]['LOO_RMST_SEP'].var()
                        var_active = df.loc[df['group']==1]['LOO_RMST_SEP'].var()

                        # Compute classic estimator
                        ate_rmst_manual = df[df['group'] == 1]['LOO_RMST_SEP'].mean() - df[df['group'] == 0]['LOO_RMST_SEP'].mean()
                        var_ate_rmst_manual = var_active/n + var_placebo/n
                
                        i = 0.6
                        df_tmp = df[df['noise'] == round(i,2)].copy()
                        kmf_trt_pred = KaplanMeierFitter().fit(df_tmp[df_tmp['group']==1]['time_pred'], event_observed=df_tmp[df_tmp['group']==1]['event_pred'])
                        kmf_con_pred = KaplanMeierFitter().fit(df_tmp[df_tmp['group']==0]['time_pred'], event_observed=df_tmp[df_tmp['group']==0]['event_pred'])
                        rmst_active_pred = restricted_mean_survival_time(kmf_trt_pred, t=1.5, return_variance=False) 
                        rmst_placebo_pred = restricted_mean_survival_time(kmf_con_pred, t=1.5, return_variance=False)
                        df_tmp.loc[df_tmp['group'] == 1, 'PREDS_LOO_RMST_WEIBULL'] = loo_rmst(df_tmp[df_tmp['group'] == 1], rmst_active_pred, follow_up, time='time_pred', event='event_pred')
                        df_tmp.loc[df_tmp['group'] == 0, 'PREDS_LOO_RMST_WEIBULL'] = loo_rmst(df_tmp[df_tmp['group'] == 0], rmst_placebo_pred, follow_up, time='time_pred', event='event_pred')
                        df_tmp['noise_level'] = round(i,2)
                        
                        # Compute PPCT estimator
                        sigma_f_2 = df_tmp['PREDS_LOO_RMST_WEIBULL'].var()
                        sigma_t_2 = var_active
                        sigma_c_2 = var_placebo
                        rho_t = np.cov(df_tmp[df_tmp['group'] == 1]['PREDS_LOO_RMST_WEIBULL'], df_tmp[df_tmp['group'] == 1]['LOO_RMST_SEP'])[0, 1]/np.sqrt(sigma_f_2*sigma_t_2)
                        rho_c = np.cov(df_tmp[df_tmp['group'] == 0]['PREDS_LOO_RMST_WEIBULL'], df_tmp[df_tmp['group'] == 0]['LOO_RMST_SEP'])[0, 1]/np.sqrt(sigma_f_2*sigma_c_2)
                        lambda_star_shifted_order = (n * np.sqrt(sigma_t_2) * rho_t + n * np.sqrt(sigma_c_2) * rho_c) / ((n+n) * np.sqrt(sigma_f_2))
                        var_ppi_shifted_order = (1/n) * (sigma_t_2 + (lambda_star_shifted_order**2)*sigma_f_2 - 2*lambda_star_shifted_order*np.sqrt(sigma_f_2*sigma_t_2)*rho_t)  + \
                                ((1/n) * (sigma_c_2 + (lambda_star_shifted_order**2)*sigma_f_2 - 2*lambda_star_shifted_order*np.sqrt(sigma_f_2*sigma_c_2)*rho_c))
                        ate_ppi_shifted_order = (df_tmp[df_tmp['group'] == 1]['LOO_RMST_SEP']- lambda_star_shifted_order* df_tmp[df_tmp['group'] == 1]['PREDS_LOO_RMST_WEIBULL']).mean() - \
                                (df_tmp[df_tmp['group'] == 0]['LOO_RMST_SEP']- lambda_star_shifted_order* df_tmp[df_tmp['group'] == 0]['PREDS_LOO_RMST_WEIBULL']).mean()
                        r_2_shifted_order = df_tmp['LOO_RMST_SEP'].corr(df_tmp['PREDS_LOO_RMST_WEIBULL'])**2
                        var_r2_shifted_order = var_ate_rmst_manual * (1- r_2_shifted_order)

                        estimations.append({
                                "n": n,
                                "sim": file.split('_')[1].split('.')[0],
                                "noise_level": round(i,2),
                                "ate_classic RMST": ate_rmst_manual,
                                "var_classic RMST": var_ate_rmst_manual,
                                "ate_ppi_shifted_order RMST": ate_ppi_shifted_order,
                                "var_ppi_shifted_order RMST": var_ppi_shifted_order,
                                "r2_shifted_order RMST": r_2_shifted_order,
                                "lambda_star_shifted_order RMST": lambda_star_shifted_order,
                                "var_ppi_with_r2_formula_shifted_order RMST": var_r2_shifted_order,
                                })
                        df_sim.append(df_tmp)

        df_sim = pd.concat(df_sim, ignore_index=True)
        estimations = pd.DataFrame(estimations)
        df_sim.to_csv(f"sim_outputs_1000_censore_weibull/df_sim_fc{fc}_ve{ve}_censpred{censore_rate_pred}.csv", index=False)
        estimations.to_csv(f"sim_outputs_1000_censore_weibull/estimations_fc{fc}_ve{ve}_censpred{censore_rate_pred}.csv", index=False)


## Simulate df with varying predicted censore rate

In [ ]:
n_values =[30, 100, 500]
ve = 30
follow_up = 1.5
fc = 0.4

for censore_rate_pred in [0.05, 0.2]: # 0.1 already done
        df_sim = []
        estimations = []
        for n in n_values:
                folder_path = f"./sim_ct_df_with_censore_weibull/sim_n{n}_fc{fc}_ve{ve}_censpred{censore_rate_pred}"
                files = sorted(os.listdir(folder_path))
                for file in tqdm(files, desc=f"Simulations for n={n} and vaccine efficacy={ve}%"):
                        df = pd.read_csv(os.path.join(folder_path, file))
                        df["n"] = n
                        df["sim"] = file.split('_')[1].split('.')[0]

                        # Compute outcome: leave-one-out RMST for each patient
                        kmf_trt = KaplanMeierFitter().fit(df[df['group']==1]['time_obs'], event_observed=df[df['group']==1]['event_obs'])
                        kmf_con = KaplanMeierFitter().fit(df[df['group']==0]['time_obs'], event_observed=df[df['group']==0]['event_obs'])
                        rmst_placebo  = restricted_mean_survival_time(kmf_con, t=1.5, return_variance=False)
                        rmst_active = restricted_mean_survival_time(kmf_trt, t=1.5, return_variance=False)

                        df.loc[df['group'] == 1, 'LOO_RMST_SEP'] = loo_rmst(df[df['group'] == 1], rmst_active, follow_up, time='time_obs', event='event_obs')
                        df.loc[df['group'] == 0, 'LOO_RMST_SEP'] = loo_rmst(df[df['group'] == 0], rmst_placebo, follow_up, time='time_obs', event='event_obs')
                        var_placebo = df.loc[df['group']==0]['LOO_RMST_SEP'].var()
                        var_active = df.loc[df['group']==1]['LOO_RMST_SEP'].var()

                        # Compute classic estimator
                        ate_rmst_manual = df[df['group'] == 1]['LOO_RMST_SEP'].mean() - df[df['group'] == 0]['LOO_RMST_SEP'].mean()
                        var_ate_rmst_manual = var_active/n + var_placebo/n
                
                        i = 0.6
                        df_tmp = df[df['noise'] == round(i,2)].copy()
                        kmf_trt_pred = KaplanMeierFitter().fit(df_tmp[df_tmp['group']==1]['time_pred'], event_observed=df_tmp[df_tmp['group']==1]['event_pred'])
                        kmf_con_pred = KaplanMeierFitter().fit(df_tmp[df_tmp['group']==0]['time_pred'], event_observed=df_tmp[df_tmp['group']==0]['event_pred'])
                        rmst_active_pred = restricted_mean_survival_time(kmf_trt_pred, t=1.5, return_variance=False) 
                        rmst_placebo_pred = restricted_mean_survival_time(kmf_con_pred, t=1.5, return_variance=False)
                        df_tmp.loc[df_tmp['group'] == 1, 'PREDS_LOO_RMST_WEIBULL'] = loo_rmst(df_tmp[df_tmp['group'] == 1], rmst_active_pred, follow_up, time='time_pred', event='event_pred')
                        df_tmp.loc[df_tmp['group'] == 0, 'PREDS_LOO_RMST_WEIBULL'] = loo_rmst(df_tmp[df_tmp['group'] == 0], rmst_placebo_pred, follow_up, time='time_pred', event='event_pred')
                        df_tmp['noise_level'] = round(i,2)

                        # Compute PPCT estimator
                        sigma_f_2 = df_tmp['PREDS_LOO_RMST_WEIBULL'].var()
                        sigma_t_2 = var_active
                        sigma_c_2 = var_placebo
                        rho_t = np.cov(df_tmp[df_tmp['group'] == 1]['PREDS_LOO_RMST_WEIBULL'], df_tmp[df_tmp['group'] == 1]['LOO_RMST_SEP'])[0, 1]/np.sqrt(sigma_f_2*sigma_t_2)
                        rho_c = np.cov(df_tmp[df_tmp['group'] == 0]['PREDS_LOO_RMST_WEIBULL'], df_tmp[df_tmp['group'] == 0]['LOO_RMST_SEP'])[0, 1]/np.sqrt(sigma_f_2*sigma_c_2)
                        lambda_star_shifted_order = (n * np.sqrt(sigma_t_2) * rho_t + n * np.sqrt(sigma_c_2) * rho_c) / ((n+n) * np.sqrt(sigma_f_2))
                        var_ppi_shifted_order = (1/n) * (sigma_t_2 + (lambda_star_shifted_order**2)*sigma_f_2 - 2*lambda_star_shifted_order*np.sqrt(sigma_f_2*sigma_t_2)*rho_t)  + \
                                ((1/n) * (sigma_c_2 + (lambda_star_shifted_order**2)*sigma_f_2 - 2*lambda_star_shifted_order*np.sqrt(sigma_f_2*sigma_c_2)*rho_c))
                        ate_ppi_shifted_order = (df_tmp[df_tmp['group'] == 1]['LOO_RMST_SEP']- lambda_star_shifted_order* df_tmp[df_tmp['group'] == 1]['PREDS_LOO_RMST_WEIBULL']).mean() - \
                                (df_tmp[df_tmp['group'] == 0]['LOO_RMST_SEP']- lambda_star_shifted_order* df_tmp[df_tmp['group'] == 0]['PREDS_LOO_RMST_WEIBULL']).mean()
                        r_2_shifted_order = df_tmp['LOO_RMST_SEP'].corr(df_tmp['PREDS_LOO_RMST_WEIBULL'])**2
                        var_r2_shifted_order = var_ate_rmst_manual * (1- r_2_shifted_order)

                        estimations.append({
                                "n": n,
                                "sim": file.split('_')[1].split('.')[0],
                                "noise_level": round(i,2),
                                "ate_classic RMST": ate_rmst_manual,
                                "var_classic RMST": var_ate_rmst_manual,
                                "ate_ppi_shifted_order RMST": ate_ppi_shifted_order,
                                "var_ppi_shifted_order RMST": var_ppi_shifted_order,
                                "r2_shifted_order RMST": r_2_shifted_order,
                                "lambda_star_shifted_order RMST": lambda_star_shifted_order,
                                "var_ppi_with_r2_formula_shifted_order RMST": var_r2_shifted_order,
                                })
                        df_sim.append(df_tmp)

        df_sim = pd.concat(df_sim, ignore_index=True)
        estimations = pd.DataFrame(estimations)
        df_sim.to_csv(f"sim_outputs_1000_censore_weibull/df_sim_fc{fc}_ve{ve}_censpred{censore_rate_pred}.csv", index=False)
        estimations.to_csv(f"sim_outputs_1000_censore_weibull/estimations_fc{fc}_ve{ve}_censpred{censore_rate_pred}.csv", index=False)


## Simulate graphs

In [ ]:
n_values = [30, 100, 500]
fc = 0.4
follow_up = 1.5
ve = 30
censore_rate_pred = 0.1 

df_sim = []
estimations = []
for n in n_values:
        folder_path = f"./sim_ct_df_with_censore_weibull/sim_n{n}_fc{fc}_ve{ve}_censpred{censore_rate_pred}"
        files = sorted(os.listdir(folder_path))
        for file in tqdm(files, desc=f"Simulations for n={n} and vaccine efficacy={ve}%"):
                df = pd.read_csv(os.path.join(folder_path, file))
                df["n"] = n
                df["sim"] = file.split('_')[1].split('.')[0]

                # Compute outcome: leave-one-out RMST for each patient
                kmf_trt = KaplanMeierFitter().fit(df[df['group']==1]['time_obs'], event_observed=df[df['group']==1]['event_obs'])
                kmf_con = KaplanMeierFitter().fit(df[df['group']==0]['time_obs'], event_observed=df[df['group']==0]['event_obs'])
                rmst_placebo  = restricted_mean_survival_time(kmf_con, t=1.5, return_variance=False)
                rmst_active = restricted_mean_survival_time(kmf_trt, t=1.5, return_variance=False)

                df.loc[df['group'] == 1, 'LOO_RMST_SEP'] = loo_rmst(df[df['group'] == 1], rmst_active, follow_up, time='time_obs', event='event_obs')
                df.loc[df['group'] == 0, 'LOO_RMST_SEP'] = loo_rmst(df[df['group'] == 0], rmst_placebo, follow_up, time='time_obs', event='event_obs')
                var_placebo = df.loc[df['group']==0]['LOO_RMST_SEP'].var()
                var_active = df.loc[df['group']==1]['LOO_RMST_SEP'].var()

                # Compute classic estimator
                ate_rmst_manual = df[df['group'] == 1]['LOO_RMST_SEP'].mean() - df[df['group'] == 0]['LOO_RMST_SEP'].mean()
                var_ate_rmst_manual = var_active/n + var_placebo/n
        
                for i in np.arange(0, 2.2, 0.2):
                    df_tmp = df[df['noise'] == round(i,2)].copy()
                    kmf_trt_pred = KaplanMeierFitter().fit(df_tmp[df_tmp['group']==1]['time_pred'], event_observed=df_tmp[df_tmp['group']==1]['event_pred'])
                    kmf_con_pred = KaplanMeierFitter().fit(df_tmp[df_tmp['group']==0]['time_pred'], event_observed=df_tmp[df_tmp['group']==0]['event_pred'])
                    rmst_active_pred = restricted_mean_survival_time(kmf_trt_pred, t=1.5, return_variance=False) 
                    rmst_placebo_pred = restricted_mean_survival_time(kmf_con_pred, t=1.5, return_variance=False)
                    df_tmp.loc[df_tmp['group'] == 1, 'PREDS_LOO_RMST_WEIBULL'] = loo_rmst(df_tmp[df_tmp['group'] == 1], rmst_active_pred, follow_up, time='time_pred', event='event_pred')
                    df_tmp.loc[df_tmp['group'] == 0, 'PREDS_LOO_RMST_WEIBULL'] = loo_rmst(df_tmp[df_tmp['group'] == 0], rmst_placebo_pred, follow_up, time='time_pred', event='event_pred')
                    df_tmp['noise_level'] = round(i,2)
                    
                    # Compute PPCT estimator
                    sigma_f_2 = df_tmp['PREDS_LOO_RMST_WEIBULL'].var()
                    sigma_t_2 = var_active
                    sigma_c_2 = var_placebo
                    rho_t = np.cov(df_tmp[df_tmp['group'] == 1]['PREDS_LOO_RMST_WEIBULL'], df_tmp[df_tmp['group'] == 1]['LOO_RMST_SEP'])[0, 1]/np.sqrt(sigma_f_2*sigma_t_2)
                    rho_c = np.cov(df_tmp[df_tmp['group'] == 0]['PREDS_LOO_RMST_WEIBULL'], df_tmp[df_tmp['group'] == 0]['LOO_RMST_SEP'])[0, 1]/np.sqrt(sigma_f_2*sigma_c_2)
                    lambda_star_shifted_order = (n * np.sqrt(sigma_t_2) * rho_t + n * np.sqrt(sigma_c_2) * rho_c) / ((n+n) * np.sqrt(sigma_f_2))
                    var_ppi_shifted_order = (1/n) * (sigma_t_2 + (lambda_star_shifted_order**2)*sigma_f_2 - 2*lambda_star_shifted_order*np.sqrt(sigma_f_2*sigma_t_2)*rho_t)  + \
                            ((1/n) * (sigma_c_2 + (lambda_star_shifted_order**2)*sigma_f_2 - 2*lambda_star_shifted_order*np.sqrt(sigma_f_2*sigma_c_2)*rho_c))
                    ate_ppi_shifted_order = (df_tmp[df_tmp['group'] == 1]['LOO_RMST_SEP']- lambda_star_shifted_order* df_tmp[df_tmp['group'] == 1]['PREDS_LOO_RMST_WEIBULL']).mean() - \
                            (df_tmp[df_tmp['group'] == 0]['LOO_RMST_SEP']- lambda_star_shifted_order* df_tmp[df_tmp['group'] == 0]['PREDS_LOO_RMST_WEIBULL']).mean()
                    r_2_shifted_order = df_tmp['LOO_RMST_SEP'].corr(df_tmp['PREDS_LOO_RMST_WEIBULL'])**2
                    var_r2_shifted_order = var_ate_rmst_manual * (1- r_2_shifted_order)

                    estimations.append({
                            "n": n,
                            "sim": file.split('_')[1].split('.')[0],
                            "noise_level": round(i,2),
                            "ate_classic RMST": ate_rmst_manual,
                            "var_classic RMST": var_ate_rmst_manual,
                            "ate_ppi_shifted_order RMST": ate_ppi_shifted_order,
                            "var_ppi_shifted_order RMST": var_ppi_shifted_order,
                            "r2_shifted_order RMST": r_2_shifted_order,
                            "lambda_star_shifted_order RMST": lambda_star_shifted_order,
                            "var_ppi_with_r2_formula_shifted_order RMST": var_r2_shifted_order,
                            })
                    df_sim.append(df_tmp)

df_sim = pd.concat(df_sim, ignore_index=True)
estimations = pd.DataFrame(estimations)
df_sim.to_csv(f"sim_outputs_1000_censore_weibull/df_sim_fc{fc}_ve{ve}_censpred{censore_rate_pred}_increasing_noise.csv", index=False)
estimations.to_csv(f"sim_outputs_1000_censore_weibull/estimations_fc{fc}_ve{ve}_censpred{censore_rate_pred}_increasing_noise.csv", index=False)


## Simulate varying treatment with 5000 iterations

In [ ]:
n_values = [30, 100]
fc = 0.4
follow_up = 1.5
censore_rate_pred = 0.10

for ve in [10,20,30,40,60,80]:
        df_sim = []
        estimations = []
        for n in n_values:
                folder_path = f"./sim_ct_df_with_censore_weibull_5000/sim_n{n}_fc{fc}_ve{ve}_censpred{censore_rate_pred}"
                files = sorted(os.listdir(folder_path))
                for file in tqdm(files, desc=f"Simulations for n={n} and vaccine efficacy={ve}%"):
                        df = pd.read_csv(os.path.join(folder_path, file))
                        df["n"] = n
                        df["sim"] = file.split('_')[1].split('.')[0]

                        # Compute outcome: leave-one-out RMST for each patient
                        kmf_trt = KaplanMeierFitter().fit(df[df['group']==1]['time_obs'], event_observed=df[df['group']==1]['event_obs'])
                        kmf_con = KaplanMeierFitter().fit(df[df['group']==0]['time_obs'], event_observed=df[df['group']==0]['event_obs'])
                        rmst_placebo  = restricted_mean_survival_time(kmf_con, t=1.5, return_variance=False)
                        rmst_active = restricted_mean_survival_time(kmf_trt, t=1.5, return_variance=False)

                        df.loc[df['group'] == 1, 'LOO_RMST_SEP'] = loo_rmst(df[df['group'] == 1], rmst_active, follow_up, time='time_obs', event='event_obs')
                        df.loc[df['group'] == 0, 'LOO_RMST_SEP'] = loo_rmst(df[df['group'] == 0], rmst_placebo, follow_up, time='time_obs', event='event_obs')
                        var_placebo = df.loc[df['group']==0]['LOO_RMST_SEP'].var()
                        var_active = df.loc[df['group']==1]['LOO_RMST_SEP'].var()

                        # Compute classic estimator
                        ate_rmst_manual = df[df['group'] == 1]['LOO_RMST_SEP'].mean() - df[df['group'] == 0]['LOO_RMST_SEP'].mean()
                        var_ate_rmst_manual = var_active/n + var_placebo/n
                
                        i = 0.6
                        df_tmp = df[df['noise'] == round(i,2)].copy()
                        kmf_trt_pred = KaplanMeierFitter().fit(df_tmp[df_tmp['group']==1]['time_pred'], event_observed=df_tmp[df_tmp['group']==1]['event_pred'])
                        kmf_con_pred = KaplanMeierFitter().fit(df_tmp[df_tmp['group']==0]['time_pred'], event_observed=df_tmp[df_tmp['group']==0]['event_pred'])
                        rmst_active_pred = restricted_mean_survival_time(kmf_trt_pred, t=1.5, return_variance=False) 
                        rmst_placebo_pred = restricted_mean_survival_time(kmf_con_pred, t=1.5, return_variance=False)
                        df_tmp.loc[df_tmp['group'] == 1, 'PREDS_LOO_RMST_WEIBULL'] = loo_rmst(df_tmp[df_tmp['group'] == 1], rmst_active_pred, follow_up, time='time_pred', event='event_pred')
                        df_tmp.loc[df_tmp['group'] == 0, 'PREDS_LOO_RMST_WEIBULL'] = loo_rmst(df_tmp[df_tmp['group'] == 0], rmst_placebo_pred, follow_up, time='time_pred', event='event_pred')
                        df_tmp['noise_level'] = round(i,2)
                        
                        # Compute PPCT estimator
                        sigma_f_2 = df_tmp['PREDS_LOO_RMST_WEIBULL'].var()
                        sigma_t_2 = var_active
                        sigma_c_2 = var_placebo
                        rho_t = np.cov(df_tmp[df_tmp['group'] == 1]['PREDS_LOO_RMST_WEIBULL'], df_tmp[df_tmp['group'] == 1]['LOO_RMST_SEP'])[0, 1]/np.sqrt(sigma_f_2*sigma_t_2)
                        rho_c = np.cov(df_tmp[df_tmp['group'] == 0]['PREDS_LOO_RMST_WEIBULL'], df_tmp[df_tmp['group'] == 0]['LOO_RMST_SEP'])[0, 1]/np.sqrt(sigma_f_2*sigma_c_2)
                        lambda_star_shifted_order = (n * np.sqrt(sigma_t_2) * rho_t + n * np.sqrt(sigma_c_2) * rho_c) / ((n+n) * np.sqrt(sigma_f_2))
                        var_ppi_shifted_order = (1/n) * (sigma_t_2 + (lambda_star_shifted_order**2)*sigma_f_2 - 2*lambda_star_shifted_order*np.sqrt(sigma_f_2*sigma_t_2)*rho_t)  + \
                                ((1/n) * (sigma_c_2 + (lambda_star_shifted_order**2)*sigma_f_2 - 2*lambda_star_shifted_order*np.sqrt(sigma_f_2*sigma_c_2)*rho_c))
                        ate_ppi_shifted_order = (df_tmp[df_tmp['group'] == 1]['LOO_RMST_SEP']- lambda_star_shifted_order* df_tmp[df_tmp['group'] == 1]['PREDS_LOO_RMST_WEIBULL']).mean() - \
                                (df_tmp[df_tmp['group'] == 0]['LOO_RMST_SEP']- lambda_star_shifted_order* df_tmp[df_tmp['group'] == 0]['PREDS_LOO_RMST_WEIBULL']).mean()
                        r_2_shifted_order = df_tmp['LOO_RMST_SEP'].corr(df_tmp['PREDS_LOO_RMST_WEIBULL'])**2
                        var_r2_shifted_order = var_ate_rmst_manual * (1- r_2_shifted_order)

                        estimations.append({
                                "n": n,
                                "sim": file.split('_')[1].split('.')[0],
                                "noise_level": round(i,2),
                                "ate_classic RMST": ate_rmst_manual,
                                "var_classic RMST": var_ate_rmst_manual,
                                "ate_ppi_shifted_order RMST": ate_ppi_shifted_order,
                                "var_ppi_shifted_order RMST": var_ppi_shifted_order,
                                "r2_shifted_order RMST": r_2_shifted_order,
                                "lambda_star_shifted_order RMST": lambda_star_shifted_order,
                                "var_ppi_with_r2_formula_shifted_order RMST": var_r2_shifted_order,
                                })
                        df_sim.append(df_tmp)

        df_sim = pd.concat(df_sim, ignore_index=True)
        estimations = pd.DataFrame(estimations)
        df_sim.to_csv(f"sim_outputs_5000_censore_weibull/df_sim_fc{fc}_ve{ve}_censpred{censore_rate_pred}.csv", index=False)
        estimations.to_csv(f"sim_outputs_5000_censore_weibull/estimations_fc{fc}_ve{ve}_censpred{censore_rate_pred}.csv", index=False)


Simulations for n=100 and vaccine efficacy=20%:  63%|██████▎   | 3133/5000 [5:35:48<3:37:43,  7.00s/it] 